# Pretest — Influencer Attractiveness and AI Disclosure

Revised survey analysis from TUM's Eye Tracking and Decision Making coursework.
Group authors are listed in the README. These notebooks analyze survey responses.

Outputs are cleared for public distribution. Execute with authorized local data.
Original results and narrative conclusions require recalculation after the corrections.
The first three data rows are removed following the original notebook's convention
(two Qualtrics metadata rows and one presumed dummy response). Verify the third-row
exclusion against the original export before interpreting results.

Missing scale items produce missing composite scores. Reliability uses complete
item responses. Regression inference uses HC3 standard errors. Mediation estimates
describe statistical indirect associations and require additional assumptions for
a causal interpretation. Bootstrap intervals are evaluated directly, independently
of whether a total or direct effect is statistically significant.


In [ ]:
pretest_comparisons = []

# Structure

# 1. Install & load packages

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from scipy.stats import mannwhitneyu
from scipy.stats import pearsonr
import seaborn as sns
import matplotlib.pyplot as plt
import pingouin as pg

# 2. load data

In [ ]:
data = pd.read_csv('data/Pretest.csv')

# 3. clean data

## 3.1 delete first tree rows

In [ ]:
data_cleaned = data.drop([0,1,2])
number_rows = len(data_cleaned)
print(f"number of rows: {number_rows}")

## 3.2 delete unfished responses

In [ ]:
# check if all results if all participants finished the survey
data_cleaned['Progress'] = pd.to_numeric(data_cleaned['Progress'], errors='coerce')
data_cleaned = data_cleaned.drop(data_cleaned[data_cleaned['Progress'] != 100].index)
number_rows = len(data_cleaned)
print(f"number of rows: {number_rows}")


# 4. Analysis

## 4.1 Define functions & prepare data

In [ ]:
def cronbach_alpha(data):
    complete = data.apply(pd.to_numeric, errors="coerce").dropna()
    if len(complete) < 2 or complete.shape[1] < 2:
        return np.nan
    total_variance = complete.sum(axis=1).var(ddof=1)
    if total_variance <= 0:
        return np.nan
    k = complete.shape[1]
    return k / (k - 1) * (1 - complete.var(ddof=1).sum() / total_variance)


In [ ]:
def cohens_d(group1, group2):
    group1, group2 = group1.dropna(), group2.dropna()
    n1, n2 = len(group1), len(group2)
    pooled_sd = np.sqrt(
        ((n1 - 1) * group1.var(ddof=1) + (n2 - 1) * group2.var(ddof=1))
        / (n1 + n2 - 2)
    )
    return (group2.mean() - group1.mean()) / pooled_sd


In [ ]:
# Ensure columns are numeric (only if needed)
data_cleaned[["Q3_1", "Q3_2", "Q3_3", "Q5"]] = data_cleaned[["Q3_1", "Q3_2", "Q3_3", "Q5"]].apply(pd.to_numeric, errors='coerce')

In [ ]:
data_cleaned["DefaultQuestionBlock_DO"] = data_cleaned["DefaultQuestionBlock_DO"].astype("string").str.strip()
# Split DataFrame based on the responses
attractive_df = data_cleaned[data_cleaned['DefaultQuestionBlock_DO'] == 'Attractive']
less_attractive_df = data_cleaned[data_cleaned['DefaultQuestionBlock_DO'] == 'Less_Attractive']
least_attractive_df = data_cleaned[data_cleaned['DefaultQuestionBlock_DO'] == 'Least_Attractive']

# Display the resulting DataFrames
print("Attractive:")
print(len(attractive_df))

print("\nLess Attractive:")
print(len(less_attractive_df))

print("\nLeast Attractive:")
print(len(least_attractive_df))

## 4.2 Cronbachs Alpha

In [ ]:
attractive = data_cleaned[["Q3_1", "Q3_2", "Q3_3"]]
alpha_attractive = round(cronbach_alpha(attractive),2)
print(f"Cronbach's Alpha for Q3 (attractiveness): {alpha_attractive}")

## 4.3 Mean & Std. (attractiveness) of different types of attractiveness

In [ ]:
print(data_cleaned['DefaultQuestionBlock_DO'].unique())

In [ ]:
data_cleaned["Q3_mean"] = data_cleaned[["Q3_1", "Q3_2", "Q3_3"]].mean(axis=1, skipna=False).apply(pd.to_numeric, errors='coerce')
data_cleaned["Q3_stdev"] = np.std(data_cleaned[["Q3_1", "Q3_2", "Q3_3"]],axis=1, ddof=1)
result_mean = data_cleaned.groupby("DefaultQuestionBlock_DO")["Q3_mean"].mean()
result_stdev = data_cleaned.groupby("DefaultQuestionBlock_DO")["Q3_mean"].std(ddof=1)
print(result_mean)
print(result_stdev)

## 4.4 Check the significant difference between the groups (attractiveness)

In [ ]:
# Extract the values for each group in the ‘attractive’ column
group_attractive = data_cleaned[data_cleaned["DefaultQuestionBlock_DO"] == "Attractive"]["Q3_mean"]
group_less_attractive = data_cleaned[data_cleaned["DefaultQuestionBlock_DO"] == "Less_Attractive"]["Q3_mean"]

# Perform a t-test
t_stat, p_value = stats.ttest_ind(group_attractive.dropna(), group_less_attractive.dropna(), equal_var=False)

# Berechnung der Freiheitsgrade nach der Welch-Satterthwaite-Formel
n1 = len(group_attractive.dropna())
n2 = len(group_less_attractive.dropna())
s1 = group_attractive.std(ddof=1)
s2 = group_less_attractive.std(ddof=1)

df = ((s1**2 / n1) + (s2**2 / n2))**2 / (((s1**2 / n1)**2 / (n1 - 1)) + ((s2**2 / n2)**2 / (n2 - 1)))

# show the results
print(f"T-Statistik: {t_stat}")
print(f"P-Wert: {p_value}")
print(f"df: {df}")

# Check whether the difference is significant (e.g. at a significance level of 0.05)
if p_value < 0.05:
    print("The difference between the groups is significant.")
else:
    print("The difference between the groups is not significant.")

print(f"Cohen's d: {abs(cohens_d(group_less_attractive, group_attractive))}")
pretest_comparisons.append({"comparison": 'Attractive vs less attractive — attractiveness', "p": float(p_value)})


In [ ]:
# Extract the values for each group in the ‘attractive’ column
group_attractive = data_cleaned[data_cleaned["DefaultQuestionBlock_DO"] == "Attractive"]["Q3_mean"]
group_least_attractive = data_cleaned[data_cleaned["DefaultQuestionBlock_DO"] == "Least_Attractive"]["Q3_mean"]

# Perform a t-test
t_stat, p_value = stats.ttest_ind(group_attractive.dropna(), group_least_attractive.dropna(), equal_var=False)

# Berechnung der Freiheitsgrade nach der Welch-Satterthwaite-Formel
n1 = len(group_attractive.dropna())
n2 = len(group_least_attractive.dropna())
s1 = group_attractive.std(ddof=1)
s2 = group_least_attractive.std(ddof=1)

df = ((s1**2 / n1) + (s2**2 / n2))**2 / (((s1**2 / n1)**2 / (n1 - 1)) + ((s2**2 / n2)**2 / (n2 - 1)))

# show the results
print(f"T-Statistik: {t_stat}")
print(f"P-Wert: {p_value}")
print(f"df: {df}")

# Check whether the difference is significant (e.g. at a significance level of 0.05)
if p_value < 0.05:
    print("The difference between the groups is significant.")
else:
    print("The difference between the groups is not significant.")

print(f"Cohen's d: {abs(cohens_d(group_least_attractive, group_attractive))}")
pretest_comparisons.append({"comparison": 'Attractive vs least attractive — attractiveness', "p": float(p_value)})


## 4.5 Mean & Std. (edited) of different types of attractiveness

In [ ]:
result_mean = data_cleaned.groupby("DefaultQuestionBlock_DO")["Q5"].mean()
result_stdev = data_cleaned.groupby("DefaultQuestionBlock_DO")["Q5"].std(ddof=1)
print(result_mean)
print(result_stdev)

## 4.6 Check the significant difference between the groups (edited)

In [ ]:
# Extract the values for each group in the ‘attractive’ column
group_attractive = data_cleaned[data_cleaned["DefaultQuestionBlock_DO"] == "Attractive"]["Q5"]
group_less_attractive = data_cleaned[data_cleaned["DefaultQuestionBlock_DO"] == "Less_Attractive"]["Q5"]

# Perform a t-test
t_stat, p_value = stats.ttest_ind(group_attractive.dropna(), group_less_attractive.dropna(), equal_var=False)

# Berechnung der Freiheitsgrade nach der Welch-Satterthwaite-Formel
n1 = len(group_attractive.dropna())
n2 = len(group_less_attractive.dropna())
s1 = group_attractive.std(ddof=1)
s2 = group_less_attractive.std(ddof=1)

df = ((s1**2 / n1) + (s2**2 / n2))**2 / (((s1**2 / n1)**2 / (n1 - 1)) + ((s2**2 / n2)**2 / (n2 - 1)))

# show the results
print(f"T-Statistik: {t_stat}")
print(f"P-Wert: {p_value}")
print(f"df: {df}")

# Check whether the difference is significant (e.g. at a significance level of 0.05)
if p_value < 0.05:
    print("The difference between the groups is significant.")
else:
    print("The difference between the groups is not significant.")

print(f"Cohen's d: {abs(cohens_d(group_less_attractive, group_attractive))}")
pretest_comparisons.append({"comparison": 'Attractive vs less attractive — editing', "p": float(p_value)})


In [ ]:
# Extract the values for each group in the ‘attractive’ column
group_attractive = data_cleaned[data_cleaned["DefaultQuestionBlock_DO"] == "Attractive"]["Q5"]
group_least_attractive = data_cleaned[data_cleaned["DefaultQuestionBlock_DO"] == "Least_Attractive"]["Q5"]

# Perform a t-test
t_stat, p_value = stats.ttest_ind(group_attractive.dropna(), group_least_attractive.dropna(), equal_var=False)

# Berechnung der Freiheitsgrade nach der Welch-Satterthwaite-Formel
n1 = len(group_attractive.dropna())
n2 = len(group_least_attractive.dropna())
s1 = group_attractive.std(ddof=1)
s2 = group_least_attractive.std(ddof=1)

df = ((s1**2 / n1) + (s2**2 / n2))**2 / (((s1**2 / n1)**2 / (n1 - 1)) + ((s2**2 / n2)**2 / (n2 - 1)))

# show the results
print(f"T-Statistik: {t_stat}")
print(f"P-Wert: {p_value}")
print(f"df: {df}")

# Check whether the difference is significant (e.g. at a significance level of 0.05)
if p_value < 0.05:
    print("The difference between the groups is significant.")
else:
    print("The difference between the groups is not significant.")

print(f"Cohen's d: {abs(cohens_d(group_least_attractive, group_attractive))}")
pretest_comparisons.append({"comparison": 'Attractive vs least attractive — editing', "p": float(p_value)})


In [ ]:
from statsmodels.stats.multitest import multipletests
comparison_table = pd.DataFrame(pretest_comparisons)
comparison_table["p_holm"] = multipletests(comparison_table["p"], method="holm")[1]
display(comparison_table)
